#### T14 Network Local HTTP Simulation

Contrast **batched HTTP** settings with T12 — batch size, retries, and when to use each.

`examples/config/network_http_batched.yaml`

**Expect:** Console + `t14_network_http_layer.jsonl` under `examples/logs/notebooks/`; batched `network_http` may fail offline. <small>More: `examples/tutorials/notebooks/README.md`.</small>


**§0 — pip**  
<small>Optional if `import hydra_logger` already works.</small>


In [1]:
%pip install -q "hydra-logger"
# Restart kernel if pip upgraded deps.


Note: you may need to restart the kernel to use updated packages.


**§1 — Setup**  
<small>Notebook plumbing (usually **collapsed**): ``importlib`` loads ``jupyter_workspace.py``, then **one call** — ``prime_notebook_workspace()`` — (``shared.path_bootstrap`` → ``utility``). Clone root follows the loaded file, not Jupyter's cwd. Presets: ``examples/config/``. Details: `notebooks/README.md`.</small>


In [2]:
import importlib.util
import os
import tempfile
from pathlib import Path

def _resolved_notebook_cwd() -> Path:
    try:
        return Path.cwd().resolve()
    except FileNotFoundError:
        fb = Path(tempfile.gettempdir()).resolve()
        os.chdir(fb)
        return fb

def _load_jupyter_workspace_module():
    _starts: list[Path] = []
    _env = os.environ.get("HYDRA_LOGGER_REPO", "").strip()
    if _env:
        _starts.append(Path(_env).expanduser().resolve())
    _starts.append(_resolved_notebook_cwd())
    _seen: set[str] = set()
    for _s in _starts:
        for _b in (_s, *_s.parents):
            _jw = _b / "examples" / "tutorials" / "jupyter_workspace.py"
            try:
                _key = str(_jw.resolve())
            except (OSError, RuntimeError):
                continue
            if _key in _seen:
                continue
            _seen.add(_key)
            if _jw.is_file():
                _spec = importlib.util.spec_from_file_location("_hydra_tutorial_jw", _jw)
                assert _spec and _spec.loader
                _mod = importlib.util.module_from_spec(_spec)
                _spec.loader.exec_module(_mod)
                return _mod
    raise FileNotFoundError(
        "Could not find examples/tutorials/jupyter_workspace.py. Clone hydra-logger, set "
        "HYDRA_LOGGER_REPO to the clone (or any directory inside it), or start Jupyter with cwd "
        "inside that clone."
    )

repo_root = _load_jupyter_workspace_module().prime_notebook_workspace()


Repo root (cwd): /home/razvansavin/Projects/hydra-logger


**§2 — Imports**

In [3]:
from hydra_logger import create_sync_logger


**§3 — Config path**  
<small>`CONFIG_PATH` is ``repo_root / "examples" / "config" / …`` — the same files you edit in the repository.</small>


In [4]:
CONFIG_PATH = repo_root / "examples" / "config" / "network_http_batched.yaml"


**§5 — Scenario**  
<small>Your hydra-logger usage pattern.</small>


In [5]:
with create_sync_logger(
    config_path=CONFIG_PATH,
    strict_unknown_fields=True,
    name="tutorial-t14",
) as logger:
    logger.info("batched", layer="ship", context={"batch_id": "b14"})


| 2026-03-21 17:20:10 | INFO | ship | batched


**§6 — Results**  
<small>Console + `t14_network_http_layer.jsonl` under `examples/logs/notebooks/`; batched `network_http` may fail offline.</small>


**Iterate:** edit YAML under `examples/config/`, re-run **§5** (and **§6** if present). <small>Details: `notebooks/README.md`.</small>
